In [11]:
import glob
from langchain_community.document_loaders import PyPDFLoader

docs = []
for path in glob.glob("data/*.pdf")[:3]:
    docs += PyPDFLoader(path).load()
print("pages loaded:",  len(docs))


pages loaded: 17


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 200 , chunk_overlap = 50)
chunks = splitter.split_documents(docs)
print("chunks:", len(chunks))
print(chunks[0].page_content[:300])

chunks: 153
1  of  3 
COOS 
10-MARK THEORY QUESTIONS 
Ordered CO-wise, CO1 to CO6 
15 pure theory  +  1 hybrid  ·  8 papers 
 
Where ten marks live


In [15]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

emb = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(chunks, emb)
retriever = db.as_retriever(search_kwargs = {"k":3})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1401.49it/s]


In [16]:
question = "Explain disk scheduling algorithms"
hits = retriever.invoke(question)
for h in hits:
    print(h.metadata.get("source"), "| page", h.metadata.get("page"))
    print(h.page_content[:300], "\n")

data\COOS_Ten_Mark_Theory.pdf | page 2
K2   What are disk scheduling algorithms? Describe SCAN and C-SCAN algorithm with example. 
25/11/2025  ·  Q3(b)  ·  10 marks  ·  paper CO6 → COOS CO6 

data\COOS_Ten_Mark_Theory.pdf | page 2
algorithms without a worked sweep earns about half. 
K1   Write a short note on Disk scheduling Algorithm. 
24/05/2024 DYP  ·  Q3(a)  ·  10 marks  ·  paper CO6 → COOS CO6 

data\COOS_Ten_Mark_Theory.pdf | page 2
K3   Illustrate with examples various Disk Scheduling Algorithms FCFS, SSTF and SCAN. 
30/04/2026  ·  Q3(c)  ·  10 marks  ·  paper CO6 → COOS CO6 
 
What the ordering tells you 



In [17]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3.2")

def answer(question):
    hits = retriever.invoke(question)
    context = "\n\n".join(h.page_content for h in hits)
    prompt = (
        "Anser the questions using ONLY the context below. "
        "If the answer is not in the context, say you dont know.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    return llm.invoke(prompt).content

print(answer("what is the capital of INDIA"))


I don't know. The context only talks about entrepreneurship in India, but it doesn't mention the capital of India.
